# Introdcution to scikit-learn (sklearn)
What we're going to learn:
0. End-to-end scikit-learn workflow
1. Getting the data ready
2. Choose the right estimator/algo for our problems
3. Fit the model/algo and use it to make predictions on our data
4. Evaluating a model
5. Improve a model
6. Save and load a trained model
7. Putting it all together

## 0. End-to-end scikit-learn workflow

In [ ]:
# Get the data ready
import pandas as pd
import numpy as np

heart_disease = pd.read_csv('../matplotlib/heart-disease.csv')
heart_disease

In [ ]:
# Create X (features matrix)
X = heart_disease.drop('target', axis=1)
# Create y (target)
y = heart_disease['target']

In [ ]:
import warnings
# 'warnings' is used to set a warning filter
warnings.filterwarnings('default')

# 2. Choose the right model and hyperparameters
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100)

# We'll keep the default hyperparameters
clf.get_params()

In [ ]:
# 3, Fit the model and make predictions
from sklearn.model_selection import train_test_split

# Take a 20% of our data for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Use the training data to fit the model
clf.fit(X_train, y_train)
print(clf.score(X_test, y_test))

In [ ]:
# Make a prediction
y_pred = clf.predict(X_test)
y_pred

In [ ]:
# 4. Evaluate the model on the training data and test data (accuracy)
clf.score(X_train, y_train), clf.score(X_test, y_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# This will print a classification report which is a summary of our model performance on the test data set (accuracy, precision, recall, f1 score)
print(classification_report(y_test, y_pred))

In [ ]:
# COnfusion matrix is a table that shows the number of true positives, true negatives, false positives and false negatives
confusion_matrix(y_test, y_pred)

In [ ]:
# Accuracy score is the ratio of the number of correct predictions to the total number of predictions
accuracy_score(y_test, y_pred)

In [ ]:
# 5. Improve a model
# Try different amount of estimators

np.random.seed(42)
for i in range(10, 100, 10):
    print(f'Trying model with {i} estimators...')
    clf = RandomForestClassifier(n_estimators=i).fit(X_train, y_train)
    print(f'Train accuracy is: {clf.score(X_train, y_train)}')
    print(f'Test accuracy is: {clf.score(X_test, y_test)}')
    print('')

In [ ]:
# 6. Save a model and load it 
import pickle

# Pickle is a way to serialize objects in Python, which means converting them into a byte stream that can be saved to a file or sent over a network.
# Here 'wb' means write binary
pickle.dump(clf, open('random_forest_model_1.pkl', 'wb'))


In [ ]:
# Here 'rb' means read binary
loaded_model = pickle.load(open('random_forest_model_1.pkl', 'rb'))
loaded_model.score(X_test, y_test)

---

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Getting our data ready to be used with machine learning
Three main things we have to do:
1. Split the data into features and labels (usually 'X' and 'y')
2. Filling (also called imputing) or disregarding missing values
3. Converting non-numerical values to numerical values (also called feature encoding)

In [ ]:
heart_disease.head()

In [ ]:
X = heart_disease.drop('target', axis=1)
X.head()

In [ ]:
y = heart_disease['target']
y.head()

In [ ]:
# Split the data into training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X.shape

### 1.1 Make sure it's all numerical

In [ ]:
car_sales = pd.read_csv('data/car-sales-extended.csv')
car_sales.head()

In [ ]:
len(car_sales)

In [ ]:
car_sales.dtypes

In [ ]:
# split into x and y
X = car_sales.drop('Price', axis=1)
y = car_sales['Price']

# split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Build machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
# model.fit(X_train, y_train)
# model.score(X_test, y_test)

# This will give an error because some stuff in X are not numbers and so the model can't really understand it

In [ ]:
# Turn the categories into numbers
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ["Make", "Colour", "Doors"] # We treat doors as categorical, for example '4 doors' is one category, and '5 doors' is another category etc

one_hot = OneHotEncoder()

transformer = ColumnTransformer([("one_hot", one_hot, categorical_features)], remainder="passthrough")

transformed_X = transformer.fit_transform(X)
transformed_X

In [ ]:
X.head()

In [ ]:
pd.DataFrame(transformed_X)

In [ ]:
# Another way to turn the categories into numbers
dummies = pd.get_dummies(car_sales[["Make", "Colour", "Doors"]])
dummies

In [ ]:
# Let's refit our model
np.random.seed(42)

X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)
model.fit(X_train, y_train)
model.score(X_test, y_test)

### 1.2 What if there were missing values?
1. Fill them with some value (also known as imputation)
2. REmove the samples with missing data altogether.

In [ ]:
# import car sales missing data
car_sales_missing = pd.read_csv('data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
# Create X and y
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

#### option 1: Fill missing data with Pandas

In [ ]:
car_sales_missing.fillna({
    "Make": "missing",
    "Colour": "missing",
    "Odometer (KM)": car_sales_missing["Odometer (KM)"].mean(),
    "Doors": 4
}, inplace=True)
car_sales_missing.isna().sum()

In [ ]:
# Remove rows with missing price value
car_sales_missing.dropna(subset=["Price"], inplace=True)
car_sales_missing.isna().sum()

In [ ]:
len(car_sales_missing)

In [ ]:
# Turn the categories into numbers
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

categorical_features = ["Make", "Colour", "Doors"] # We treat doors as categorical, for example '4 doors' is one category, and '5 doors' is another category etc

one_hot = OneHotEncoder()

transformer = ColumnTransformer([("one_hot", one_hot, categorical_features)], remainder="passthrough")

transformed_X = transformer.fit_transform(car_sales_missing)
transformed_X

### Option 2: Fill missing values with scikit-learn

In [ ]:
car_sales_missing = pd.read_csv('data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
# Drop the rows with no labels
car_sales_missing.dropna(subset=["Price"], inplace=True)
car_sales_missing.isna().sum()

In [ ]:
# Split into X and y
X = car_sales_missing.drop("Price", axis = 1)
y = car_sales_missing["Price"]

In [ ]:
# Fill missing values with Scikit-Learn
from sklearn.impute import SimpleImputer

# Fill categorical valyes with 'missing' & numeric values with mean
cat_imputer = SimpleImputer(strategy = "constant", fill_value = "missing")
door_imputer = SimpleImputer(strategy = "constant", fill_value = 4)
num_imputer = SimpleImputer(strategy = "mean")

# Define cokumns
cat_features = ["Make", "Colour"]
door_features = ["Doors"]
num_features = ["Odometer (KM)"]

# Create the imputer
imputer = ColumnTransformer([
    ("cat_imputer", cat_imputer, cat_features),
    ("door_imputer", door_imputer, door_features),
    ("num_imputer", num_imputer, num_features)
])
imputed_X = imputer.fit_transform(X)
imputed_X

In [ ]:
car_sales_filled = pd.DataFrame(imputed_X, columns = ["Make", "Colour", "Doors", "Odometer (KM)"])
car_sales_filled.head()

In [ ]:
# Turn the categories into numbers
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# X = car_sales_missing.drop('Price', axis=1)
# y = car_sales_missing['Price']

categorical_features = ["Make", "Colour", "Doors"] # We treat doors as categorical, for example '4 doors' is one category, and '5 doors' is another category etc

one_hot = OneHotEncoder()

transformer = ColumnTransformer([("one_hot", one_hot, categorical_features)], remainder="passthrough")

transformed_X = transformer.fit_transform(car_sales_filled)
transformed_X

In [ ]:
# Now we got our data as numbers filled (no missing values)
# Let's fit a model
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
len(car_sales_filled), len(car_sales)

## 2. Choosing the right estimator/ago for your probelm
Some things to note:
* Some things to note:
* Sklearn refers to machine learning models, algos as estimators
* Classification problem - predicting a category (ex: heart disease or not)
    * Sometimes you'll see `CLF` (short for classifier) used as a classification estimator
* Regression probelm - predicting a number (selling price of a car)

Check this link for more info:
<a href="https://scikit-learn.org/stable/machine_learning_map.html">SciKit learn machine learning map</a>

### 2.1 Picking a machine learning model for a regression problem
let's use the California housing dataset

In [ ]:
# Get california housing dataset
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
housing

In [ ]:
import pandas as pd
housing_df = pd.DataFrame(housing["data"], columns=housing["feature_names"])
housing_df.head()

In [ ]:
housing_df["target"] = housing["target"]
housing_df.head()

In [ ]:
# import algorithm
from sklearn.linear_model import Ridge  

# Setup random seed
np.random.seed(42)

# Create the data
X = housing_df.drop("target", axis=1)
y = housing_df["target"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate and fit the model (on the training set)
model = Ridge()
model.fit(X_train, y_train)

# Check the score of the model (on the test set)
model.score(X_test, y_test)


What if `Ridge` didn't work or the score didn't fit our needs?
We coudl always try a different model.
We can try now an ensemble model (an ensemble is a combinatin of smaller models to try and make a better decision that just a single model)

In [ ]:
# Import the random forest model from the ensemble module
from sklearn.ensemble import RandomForestRegressor

# Setup random seed
np.random.seed(42)

# Create the data
X = housing_df.drop("target", axis=1)
y = housing_df["target"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create random forest model
model = RandomForestRegressor()
model.fit(X_train, y_train)

# Check the score of the model (on the test set)
model.score(X_test, y_test)

## 2.2 Picking a machine learning algo for a classification problem

In [ ]:
heart_disease = pd.read_csv('data/heart-disease.csv')
heart_disease.head()

In [ ]:
# Import the linearSVC estimator class
from sklearn.svm import LinearSVC

# Setup random seed
np.random.seed(42)

# Make the data
X = heart_disease.drop("target", axis=1)
y = heart_disease["target"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate LinearSVC
clf = LinearSVC(max_iter=10000)
clf.fit(X_train, y_train)

# Evaluate the LinearSVC
clf.score(X_test, y_test)


In [ ]:
# Import the linearSVC estimator class
from sklearn.ensemble import RandomForestClassifier

# Setup random seed
np.random.seed(42)

# Make the data
X = heart_disease.drop("target", axis=1)
y = heart_disease["target"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate random forest classifier
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

# Evaluate the random forest classifier
clf.score(X_test, y_test)


## 3. Fit the model/algo on our data and use it to make predictions

### 3.1 Fit the mode to the data
Different names for: 

X = features, features variables, data

y = labels, targets, target variables

In [ ]:
# Import the linearSVC estimator class
from sklearn.ensemble import RandomForestClassifier

# Setup random seed
np.random.seed(42)

# Make the data
X = heart_disease.drop("target", axis=1)
y = heart_disease["target"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate random forest classifier
clf = RandomForestClassifier()

# First the model to the data
clf.fit(X_train, y_train)

# Evaluate the random forest classifier
clf.score(X_test, y_test)

In [ ]:
X.head()

In [ ]:
y.head()

### 3.2 Make predictions using a machine learning model